In [8]:
import os
import sys
import ast
from typing import List, Any
import random
import inspect
from tqdm import tqdm

In [9]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
sys.path.append(par_dir)

In [10]:
from llm_models.code_llms import Mistral
from database import MongoDBHelper
from tester import LLMTest
from utility.mutation_functions import CodeMutator
from prompt_templates.prompt_template import CodeInconsistencyPromptTemplate


In [11]:
db = MongoDBHelper()
if db.check_database_connectivity():
    print("MongoDB connected")

Failed to connect to MongoDB cluster, retrying again
Tries left: 4
Failed to connect to MongoDB cluster, retrying again
Tries left: 3
MongoDB connected


In [12]:
base_qns_db = db.client["Base_Questions_DB"]
question_database = base_qns_db["HumanEval_Open_Ended"]

In [13]:
tf_question_database = base_qns_db["HumanEval_Input_Output"]

In [14]:
sample_qn = question_database.find_one({"_id" : "HumanEvalo8"})
qn = sample_qn['qn']
canon_sol = sample_qn['canon_solution']
check = sample_qn['check']

complete_sol = qn + '\n' + canon_sol
print(complete_sol)

from typing import List, Tuple

def sum_product(numbers: List[int]) -> Tuple[int, int]:
    sum_value = 0
    prod_value = 1

    for n in numbers:
        sum_value += n
        prod_value *= n
    return sum_value, prod_value



In [15]:
mutated_sol = CodeMutator.mutate_for_to_enumerate(complete_sol)

In [16]:
print(mutated_sol)

from typing import List, Tuple

def sum_product(numbers: List[int]) -> Tuple[int, int]:
    sum_value = 0
    prod_value = 1
    for idx, n in enumerate(numbers):
        sum_value += n
        prod_value *= n
    return (sum_value, prod_value)


In [50]:
from typing import Tuple, Dict
class AST_Helper:

    COMPARE_OP_MAP = {
        ast.Eq: "==",
        ast.NotEq: "!=",
        ast.Lt: "<",
        ast.LtE: "<=",
        ast.Gt: ">",
        ast.GtE: ">=",
        ast.Is: "is",
        ast.IsNot: "is not",
        ast.In: "in",
        ast.NotIn: "not in",
    }

    def route_ast_type(code: str):
        if isinstance(code, ast.Call):
            return AST_Helper.extract_ast_call_args(code)

        elif isinstance(code, ast.Constant):
            return AST_Helper.extract_ast_constant(code)
        
        elif isinstance(code, ast.UnaryOp):
            return AST_Helper.extract_ast_unaryop(code)
        
        elif isinstance(code, ast.List):
            return AST_Helper.extract_ast_list(code)
        
        elif isinstance(code, list):
            res = [AST_Helper.route_ast_type(c) for c in code]
            return res[0] if len(res) == 1 else res
        
        elif isinstance(code, ast.Tuple):
            return AST_Helper.extract_ast_tuple(code)
        
        elif isinstance(code, ast.Dict):
            return AST_Helper.extract_ast_dict(code)
            
        elif isinstance(code, ast.BinOp):
            return AST_Helper.extract_ast_bin_op(code)
        
        elif isinstance(code, ast.Name):
            return AST_Helper.extract_ast_name(code)
        
        elif isinstance(code, ast.operator):
            if isinstance(code, ast.Mult):
                return "*"
            elif isinstance(code, ast.Sub):
                return "-"
            elif isinstance(code, ast.Div):
                return "/"
            elif isinstance(code, ast.Add):
                return "+"
            elif isinstance(code, ast.Pow):
                return "**"
            else:
                raise ValueError(f"A method to process {type(code)} operator type has not been developed")
        
        elif not isinstance(code, ast.AST):
            return code
        
        else:
            raise ValueError(f"A method to process {type(code)} has not been developed")

    def extract_ast_dict(code: ast.Dict) -> Dict[Any, Any]:
        if isinstance(code, ast.Dict):
            dict_values = code.values
            dict_keys = code.keys
            key_val_pairs = zip(dict_keys, dict_values)
            res = {}
            for pair in key_val_pairs:
                res[AST_Helper.route_ast_type(pair[0])] = AST_Helper.route_ast_type(pair[1])
            return res

        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Dict type.")

    def extract_ast_call_args(code: ast.Call) -> Tuple[str, list[str]]:
        if isinstance(code, ast.Call):
            test_args = [AST_Helper.route_ast_type(arg) for arg in code.args]
            if code.func.id not in dir(__builtins__):
                args_meta_data = [type(arg).__name__ for arg in code.args]
                return test_args[0] if len(test_args) == 1 else test_args, args_meta_data
            else:
                return test_args[0] if len(test_args) == 1 else test_args
        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Call type.")
        
    def extract_ast_constant(code: ast.Constant) -> str:
        if isinstance(code, ast.Constant):
            return code.value
        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Constant type.")
    
    def extract_ast_compare(code: ast.Compare) -> Tuple[Any, str | List[str], Any]:
        if isinstance(code, ast.Compare):
            left_side = code.left
            if isinstance(left_side, ast.AST):
                left = AST_Helper.route_ast_type(left_side)

            comparators = AST_Helper.route_ast_type(code.comparators)

            ops = [AST_Helper.COMPARE_OP_MAP[type(op)] for op in code.ops]                    # a list is used here as there could be more than 1 ops

            return (
                left, 
                ops[0] if len(ops) == 1 else ops, 
                comparators
            )
        
        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Constant type.")
        
    def extract_ast_tuple(code: ast.Tuple) -> Tuple[Any]:
        if isinstance(code, ast.Tuple):
            elts = code.elts
            return tuple(AST_Helper.route_ast_type(elt) for elt in elts)
        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Tuple type.")
    
    def extract_ast_bin_op(code: ast.BinOp) -> int:
        if isinstance(code, ast.BinOp):
            left = AST_Helper.route_ast_type(code.left)
            right = AST_Helper.route_ast_type(code.right)
            oper = AST_Helper.route_ast_type(code.op)

            def format_val(val):
                return repr(val) if isinstance(val, str) else val
            
            return (eval(f"{format_val(left)} {oper} {format_val(right)}"))
        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Tuple type.")

    def extract_ast_name(code: ast.Name) -> str:
        if isinstance(code, ast.Name):
            return code.id
        else:
            raise ValueError("Incorrect extraction method used, code snippet is not ast.Name type.")

    def extract_ast_unaryop(code: ast.UnaryOp) -> int | float:
        unary_map = {
            ast.USub: "-",
            ast.UAdd: "+",
            ast.Not: "not ",
            ast.Invert: "~"
        }
        if isinstance(code, ast.UnaryOp):
            symbol = unary_map[type(code.op)]
            value = str(code.operand.value)
            try:
                return int(symbol + value)
            except ValueError:
                return float(symbol + value)
            except Exception as e:
                raise ValueError(f"Unable to extract the unaryop due to the following error: {e}")

    def extract_ast_list(code: ast.List) -> List[Any]:
        if isinstance(code, ast.List):
            list_elements = code.elts
            return [AST_Helper.route_ast_type(elt) for elt in list_elements]
        else:
            print(type(code))
            raise ValueError("Incorrect extraction method used, code snippet is not ast.List type.")

In [ ]:
def extract_assert_cases(code: str):
    test_cases = []             # list storing the test parameters and test outputs for this check function
    num_cases = 0               # integer storing the number of test cases in this check function
    failed_cases = set()        # assert statements that failed to extract, if any
    rejected_cases = 0           # rejected test cases as the assert statements do not check for "=="

    tree = ast.parse(code)
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            for subnode in node.body:       #iterating through eachline node within the check function
                if isinstance(subnode, ast.Assert):
                    test_expr = subnode.test
                    num_cases+= 1
                    # try:
                    test_outputs = None
                    test_params = None

                    if isinstance(test_expr, ast.Compare):
                        ### Extracts test cases such as "assert candidate([1,2,3]) == 3"
                        ops_type = test_expr.ops[0]                 # assuming only one operator in the assert case
                        if type(ops_type) != ast.Eq:
                            rejected_cases += 1
                            num_cases -= 1                          # not collecting comparisons with "<", ">", etc as test cases
                            continue
                        else:
                            test_params, test_operators, test_outputs = AST_Helper.extract_ast_compare(test_expr)
                    elif isinstance(test_expr, ast.Call):
                        ### Extracts test cases such as assert candidate([1,2,3]) 
                        test_params = AST_Helper.extract_ast_call_args(test_expr)
                        test_outputs = True

                    elif isinstance(test_expr, ast.UnaryOp):
                        ### Extracts test cases such as assert not candidate([1,2,3])
                        op = test_expr.op
                        operand = test_expr.operand
                        test_params = AST_Helper.route_ast_type(operand)
                        if isinstance(op, ast.Not):
                            test_outputs = False 
                        else: 
                            failed_cases.add(num_cases-1)
                        
                    elif isinstance(test_expr, ast.Constant):
                        ### Ignores test cases such as assert True
                        num_cases -= 1
                        continue
                    else:
                        print("Can't decide: ", type(test_expr))
                        continue

                    test_cases.append((test_params, test_outputs))
                        
                        
                    # except Exception as e:
                    #     print("Main loop failed due to this error:", e)
    return num_cases, test_cases, rejected_cases

x = """

def check(candidate):
    assert candidate(2) == [2]
    assert candidate(3 * 19) == [3, 19]
    assert candidate(3 * 19 * 3 * 19) == [3, 3, 19, 19]
    assert candidate(3 * 19 * 3 * 19 * 3 * 19) == [3, 3, 3, 19, 19, 19]
    assert candidate(3 * 19 * 19 * 19) == [3, 19, 19, 19]
    assert candidate(3 * 2 * 3) == [2, 3, 3]
    assert candidate("<><><<<><><>><>><<><><<>>>")
    assert not candidate("<><><<><>><>>><>")
    assert candidate(["<><><<<><><>><>><<><><<>>>"])
    assert tuple(candidate([1, 2, 3])) == tuple([1, 2, 3])


    """

# x = """

# def check(candidate):
#     assert candidate([]) == (0, 1)
#     assert candidate([1, 1, 1]) == (3, 1)
#     assert candidate([100, 0]) == (100, 0)
#     assert candidate([3, 5, 7]) == (3 + 5 + 7, 3 * 5 * 7)
#     assert candidate([10]) == (10, 10)
#     assert tuple(candidate([1, 2, 3])) == tuple([1, 2, 3])
# """

num_cases, ans, rejected_cases = extract_assert_cases(x)
for i in ans:
    print(i)


((2, ['Constant']), [2])
((57, ['BinOp']), [3, 19])
((3249, ['BinOp']), [3, 3, 19, 19])
((185193, ['BinOp']), [3, 3, 3, 19, 19, 19])
((20577, ['BinOp']), [3, 19, 19, 19])
((18, ['BinOp']), [2, 3, 3])
(('<><><<<><><>><>><<><><<>>>', ['Constant']), True)
(('<><><<><>><>>><>', ['Constant']), False)
((['<><><<<><><>><>><<><><<>>>'], ['List']), True)
(([1, 2, 3], ['List']), [1, 2, 3])


In [ ]:
c = set()
tot = 0
repurposed_qn = {}
rej_tot = 0
for i in tqdm(range(question_database.count_documents({}))):
    task_id = f"HumanEvalo{i}"
    sample_qn = question_database.find_one({"_id" : task_id})
    qn = sample_qn['qn']
    qn_desc = sample_qn['qn_desc']
    examples = sample_qn['examples']
    canon_sol = sample_qn['canon_solution']
    check = sample_qn['check']
    original_id = sample_qn['original_id']

    complete_sol = qn + '\n' + canon_sol

    try:
        num_cases, test_cases, rejected_cases = extract_assert_cases(check)
        if num_cases != len(test_cases):
            print(f"{task_id}: num cases = {num_cases}, test cases extracted = {test_cases}")
        tot += num_cases
        rej_tot += rejected_cases
        if len(test_cases) < 1:
            c.add(task_id)
        else:
            repurposed_qn[task_id] = test_cases

    except Exception as e:
        print(task_id)
        print(f"Failed due to following error: {e}")

    for idx, test_case_details in enumerate(test_cases):
        test_case_id = f"HumanEvalTF{tot - len(test_cases) + idx}"

        test_case, expected_output = test_case_details
        test_input, args_meta_data = test_case

        namespace = {}

        random_test_case = random.choice(list(examples.keys()))
        func_name  = LLMTest.extract_func_name(random_test_case)

        exec(complete_sol, namespace)

        sig = inspect.signature(namespace[func_name])

        try:
            if len(sig.parameters) > 1 and isinstance(test_input, list):
                assert namespace[func_name](*test_input) == expected_output

            else:
                assert namespace[func_name](test_input) == expected_output


        except:
            print(f"Did not pass test case. Double check task_id {task_id}, test_case {test_input}")


        ### Storing / updating entry in the database
        db_entry = {
            "_id" : test_case_id,
            "full_sol" : complete_sol,
            "qn_desc": qn_desc,
            "input" : {
                "test_input": test_input,
                "input_metadata": args_meta_data,
                },
            "expected_output": expected_output,
            "examples": examples,
            "original_id": original_id
        }

        try:
            if tf_question_database.find_one({"_id": test_case_id}):
                tf_question_database.update_one(
                    filter= {"_id": test_case_id},
                    update={"$set": db_entry}
                )
            else:
                tf_question_database.insert_one(db_entry)
        except Exception as e:
            print(f"Could not enter test case {test_case_id} into TF database due to the following error: {e}")
            print(f"Testcase: {test_case}")
            
    # print(len(test_cases))


 30%|███       | 49/161 [01:54<06:26,  3.45s/it]

[<ast.List object at 0x11f829330>, <ast.Constant object at 0x11f829780>]
[<ast.List object at 0x11f828970>, <ast.Constant object at 0x11f82b220>]
[<ast.List object at 0x11f828eb0>, <ast.Constant object at 0x11f829960>]
[<ast.List object at 0x11f82a530>, <ast.Constant object at 0x11f82a1a0>]


 33%|███▎      | 53/161 [02:05<05:12,  2.89s/it]

 36%|███▌      | 58/161 [02:22<05:15,  3.06s/it]

 57%|█████▋    | 92/161 [04:38<04:46,  4.16s/it]

Could not enter test case HumanEvalTF551 into TF database due to the following error: Invalid document {'_id': 'HumanEvalTF551', 'full_sol': 'def check_dict_case(dict):\n    if len(dict.keys()) == 0:\n        return False\n    else:\n        state = "start"\n        for key in dict.keys():\n\n            if isinstance(key, str) == False:\n                state = "mixed"\n                break\n            if state == "start":\n                if key.isupper():\n                    state = "upper"\n                elif key.islower():\n                    state = "lower"\n                else:\n                    break\n            elif (state == "upper" and not key.isupper()) or (state == "lower" and not key.islower()):\n                    state = "mixed"\n                    break\n            else:\n                break\n        return state == "upper" or state == "lower" \n', 'qn_desc': 'Given a dictionary, return True if all keys are strings in lower \ncase or all keys are string

 62%|██████▏   | 100/161 [05:12<03:57,  3.90s/it]

In [20]:
print(f"{tf_question_database.count_documents({})} total test cases in the database")
print(f'{tot} valid test cases')
print(f'{rej_tot} test cases were rejected')

1110 total test cases in the database
1111 valid test cases
11 test cases were rejected


In [21]:
print(len("000052369044"))

12
